# Дообучение реранкера (cross-encoder) на датасете Adilet Legal QA

Реранкер — второй этап поиска: гибрид (дообученная e5 + BM25) отбирает топ-N чанков, cross-encoder читает
вопрос и чанк **вместе** и переупорядочивает их. Сейчас в системе стоит `BAAI/bge-reranker-v2-m3` без дообучения
(test MRR@10 0,650 → 0,714). Ноутбук дообучает его (или `mmarco-mMiniLMv2`) на наших данных и строит таблицу «до / после».

1. **Данные** — то же разбиение train / val / test по фрагментам, что в `finetune_e5.ipynb` (восстанавливается по `SEED`
   и сверяется с сохранённым), те же чанки 500 символов и эталоны `GoldChunkMatcher`.
2. **Первый этап** — Hybrid с дообученной e5 и весами из `hybrid_tuning_{lang}.json`, ровно как в `eval_reranker.py`.
3. **Hard negatives** — из топ-30 **гибрида**: реранкер в работе видит именно этих кандидатов.
4. **Baseline** — реранкер без дообучения на тех же кандидатах (должен совпасть с `reranker_eval_{lang}.json`).
5. **Дообучение** — пары (вопрос, чанк, 1/0), `BinaryCrossEntropyLoss` с `pos_weight`, выбор лучшей точки по MRR@10 на val.
6. **Сравнение «до / после»** на test, выбор размера пула на val (правило `eval_reranker.py`), сохранение результатов.

### Что нужно для Colab
- **GPU:** Runtime → Change runtime type → T4 GPU (или лучше).
- **Дообученная e5** (первый этап) на Google Drive — единственное, что нужно загрузить вручную: модель есть только
  локально (в git её нет), а без неё кандидаты будут не те, что в системе. Проще всего — архив
  `e5-small-adilet-rus.zip` (внутри папка `e5-small-adilet-rus/`) в корень `MyDrive/` или в `MyDrive/rag/`;
  можно и папкой в `MyDrive/rag/models/e5-small-adilet-rus/`.
- Всё остальное ноутбук сделает сам: склонирует проект, поставит пакеты, скачает датасет с Hugging Face.
- **Результаты** пишутся на Drive в `MyDrive/rag/` с той же структурой папок, что в проекте:
  `models/reranker-…/` (лучшая модель), `checkpoints/reranker-…/seed_N/` (журналы и метрики запусков),
  `data/adilet/reranker_finetune_{lang}.json` (отчёт). Модели отдельных запусков и промежуточные чекпоинты
  лежат на локальном диске Colab и пропадают вместе с сессией.

**Время на T4 (оценка):** первый этап ~5 мин (BM25 по 7,4 тыс. вопросов), baseline ~10 мин,
обучение `bge-reranker-v2-m3` ~20–30 мин на эпоху, `mMiniLM` — в несколько раз быстрее.
**Место на Drive:** e5 0,47 ГБ + итоговая модель: `bge-reranker-v2-m3` 2,2 ГБ, `mMiniLM` 0,47 ГБ
(обе вместе — ~3,1 ГБ). Число seed на место на Drive не влияет.

> Сначала можно включить `SMOKE_TEST = True` — короткий прогон за несколько минут, чтобы проверить, что всё работает.

In [ ]:
# --- Окружение: в Colab монтирует Drive, клонирует проект, ставит пакеты и скачивает датасет. Локально ничего не делает ---
import os
import sys
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DamirKx/rag.git"
REPO_BRANCH = "feat/web-demo-finetune"           # ветка с tune_hybrid.py и eval_reranker.py
DRIVE_DIR = Path("/content/drive/MyDrive/rag")   # исходная e5 и все результаты


def sh(cmd):
    """Запускает команду и печатает её вывод в ячейку (иначе в Jupyter он не виден)."""
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/rag/.git").exists():
        sh(f"git clone -b {REPO_BRANCH} {REPO_URL} /content/rag")
    os.chdir("/content/rag")
    # Версии как в requirements.txt; numpy / pandas / torch остаются колабовские
    sh('pip install -q "sentence-transformers==6.1.0" "rank-bm25==0.2.2" "faiss-cpu==1.15.1" "datasets==5.1.0" "accelerate==1.15.0"')
    if not Path("data/raw/adilet_legal_qa_kz.parquet").exists():
        sh("python prepare_adilet_data.py --lang all")
    # Если следующая ячейка падает на импорте после установки пакетов: Runtime → Restart session и запустить всё заново

## 0. Настройки

In [ ]:
import gc
import json
import math
import random
import time
from collections import Counter, defaultdict

# Ноутбук лежит в notebooks/, а модули проекта — в корне репозитория
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.embeddings import EmbeddingModel
from src.retriever import VectorRetriever, BM25Retriever
from src.reranker import CrossEncoderReranker
from prepare_adilet_data import clean_adilet_context
from run_adilet_eval import GoldChunkMatcher, build_fixed_chunks, get_or_compute_embeddings, model_cache_name
from tune_hybrid import EVAL_K, ir_metrics, rrf_fuse
from eval_reranker import POOL_SIZES, POOL_TOLERANCE, apply_rerank, choose_pool

In [ ]:
LANG = "rus"
BASE_RERANKER = "BAAI/bge-reranker-v2-m3"   # или "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

# Параметры обучения под модель. bge-reranker-v2-m3 (568 млн параметров) уже сильный реранкер —
# поэтому lr меньше; gradient checkpointing экономит видеопамять ценой ~30% скорости.
PRESETS = {
    "BAAI/bge-reranker-v2-m3": {"short": "bge-m3", "batch_size": 16, "grad_accum": 2, "lr": 1e-5, "grad_ckpt": True},
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1": {"short": "mminilm", "batch_size": 32, "grad_accum": 1, "lr": 2e-5, "grad_ckpt": False},
}
P = PRESETS[BASE_RERANKER]

FIRST_STAGE_MODEL = f"models/e5-small-adilet-{LANG}"   # дообученная e5 — первый этап
SEED = 42                  # seed разбиения (как в finetune_e5.ipynb) и выбора негативов
TRAIN_SEEDS = [42]         # seed обучения; [42, 43, 44] — для оценки разброса (×3 времени и места на Drive)

SPLIT_RATIOS = (0.8, 0.1, 0.1)
CHUNK_SIZE, CHUNK_OVERLAP = 500, 50
MIN_COVERAGE, TIE_MARGIN = 0.5, 0.05   # как в finetune_e5.ipynb / tune_hybrid.py

# Hard negatives из выдачи гибрида
NEG_DEPTH = 30             # негативы берутся из топ-30 гибрида
N_NEG = 7                  # негативов на вопрос (случайно из подходящих в топ-NEG_DEPTH)
NEG_MAX_COV_OTHER = 0.5    # чанк другого фрагмента с долей основ слов ответа ≥ 0.5 — вероятный ложный негатив
NEG_MAX_COV_SAME = 0.3     # для соседних чанков того же фрагмента порог строже: ответ часто лежит на границе

# Обучение
EPOCHS = 2                 # у e5 почти весь прирост дала 1-я эпоха
EVALS_PER_EPOCH = 2        # оценка на val (и кандидат в лучшую точку) дважды за эпоху
WARMUP_RATIO = 0.1
MAX_LEN = 256              # пара «вопрос + чанк» ≤ ~260 токенов, медиана чанка ~105
EVAL_POOL = 20             # val-оценщик во время обучения переранжирует топ-20 гибрида

SMOKE_TEST = False         # короткая проверка: 300 / 100 / 100 вопросов, 1 эпоха, без сохранения отчёта

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # A100/L4 — bf16, T4 — fp16

RUN_NAME = f"reranker-{P['short']}-adilet-{LANG}"
if SMOKE_TEST:
    EPOCHS, TRAIN_SEEDS, RUN_NAME = 1, TRAIN_SEEDS[:1], RUN_NAME + "-smoke"

OUT_ROOT = DRIVE_DIR if IN_COLAB else ROOT                       # результаты: та же структура, что в проекте
OUTPUT_DIR = OUT_ROOT / "models" / RUN_NAME                       # лучший по val запуск — единственная модель на Drive
LOG_DIR = OUT_ROOT / "checkpoints" / RUN_NAME                     # журналы и метрики запусков (килобайты)
# Модели всех запусков — на локальном диске Colab (там >100 ГБ), чтобы не занимать Drive
CHECKPOINT_DIR = (Path("/content/checkpoints") if IN_COLAB else ROOT / "checkpoints") / RUN_NAME
TRAINER_TMP = (Path("/content/trainer_tmp") if IN_COLAB else ROOT / "checkpoints" / "_trainer_tmp") / RUN_NAME
RESULTS_PATH = OUT_ROOT / "data" / "adilet" / f"reranker_finetune_{LANG}.json"
CACHE_DIR = "data/adilet/cache"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
gpu = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-"
print(f"Устройство: {DEVICE} ({gpu}) | {'bf16' if USE_BF16 else 'fp16' if DEVICE == 'cuda' else 'fp32'} | "
      f"реранкер: {BASE_RERANKER} | seeds: {TRAIN_SEEDS} | smoke: {SMOKE_TEST}")
print(f"Результаты: {OUT_ROOT}")

In [ ]:
# Дообученная e5 для первого этапа: в Colab берётся с Drive — папкой или zip-архивом
e5_local = Path(FIRST_STAGE_MODEL)
if not (e5_local / "config.json").exists():
    e5_drive = DRIVE_DIR / FIRST_STAGE_MODEL
    e5_zips = [p for p in [DRIVE_DIR / f"{e5_local.name}.zip", DRIVE_DIR.parent / f"{e5_local.name}.zip"] if p.exists()]
    if IN_COLAB and (e5_drive / "config.json").exists():
        shutil.copytree(e5_drive, e5_local)
        print(f"e5 скопирована с Drive: {e5_drive}")
    elif IN_COLAB and e5_zips:
        import zipfile
        # Сначала целиком копируем архив на локальный диск: распаковка прямо с Drive может прочитать файл не полностью
        local_zip = Path("/content") / e5_zips[0].name
        shutil.copy(e5_zips[0], local_zip)
        with zipfile.ZipFile(local_zip) as z:
            bad = z.testzip()                                  # проверка CRC всех файлов архива
            assert bad is None, f"Архив повреждён ({bad}) — загрузите его на Drive заново"
            z.extractall(e5_local.parent)                      # в архиве папка e5-small-adilet-{lang}/
        print(f"e5 распакована с Drive: {e5_zips[0]}")
    if not (e5_local / "config.json").exists():
        raise FileNotFoundError(
            f"Нет {FIRST_STAGE_MODEL}. Загрузите на Google Drive архив {e5_local.name}.zip (в MyDrive/ или MyDrive/rag/) "
            f"или папку {e5_local.name}/ в {e5_drive}."
        )

# Неполная копия весов даёт SafetensorError при загрузке — проверяем заголовок заранее
from safetensors import safe_open
try:
    with safe_open(str(e5_local / "model.safetensors"), framework="pt") as f:
        list(f.keys())
except Exception as e:
    raise RuntimeError(f"{e5_local}/model.safetensors повреждён или недокачан ({e}). "
                       f"Удалите папку {e5_local} и запустите ячейку снова.") from e

# Веса гибрида, подобранные tune_hybrid.py для этой модели (файл есть в git)
tuned = json.loads(Path(f"data/adilet/hybrid_tuning_{LANG}.json").read_text(encoding="utf-8"))["models"].get(e5_local.as_posix())
assert tuned, f"В hybrid_tuning_{LANG}.json нет параметров для {e5_local.as_posix()}"
HYBRID = {"dense_weight": tuned["dense_weight"], "rrf_k": tuned["rrf_k"]}
print(f"Первый этап: Hybrid, {FIRST_STAGE_MODEL}, {HYBRID}")

## 1. Данные и разбиение

Разбиение повторяет `finetune_e5.ipynb` (тот же `SEED` и алгоритм): e5 не видела val/test-фрагменты, и реранкер
их тоже не увидит. Если файлы `data/adilet/splits/` есть, разбиение сверяется с ними по id вопросов; размеры частей
сверяются с `finetune_results_{lang}.json` (он в git).

In [ ]:
df = pd.read_parquet("data/raw/adilet_legal_qa_kz.parquet")
sub_df = df[df["language"] == LANG].copy().reset_index(drop=True)
sub_df["cleaned_context"] = sub_df["context"].apply(clean_adilet_context)
sub_df["context_id"] = [f"ctx_{i:04d}" for i in range(len(sub_df))]
ctx_law = dict(zip(sub_df["context_id"], sub_df["source_title"]))

with open(f"data/adilet/qa_{LANG}_full.json", encoding="utf-8") as f:
    qa_all = json.load(f)
assert all(ctx_law[q["context_id"]] == q["source_title"] for q in qa_all), "QA не соответствует parquet — перезапустите prepare_adilet_data.py"

# То же разбиение, что в finetune_e5.ipynb
rng = np.random.default_rng(SEED)
split_of_ctx = {}
for law, group in sub_df.groupby("source_title"):
    ids = list(group["context_id"])
    rng.shuffle(ids)
    n_test = max(1, round(len(ids) * SPLIT_RATIOS[2]))
    n_val = max(1, round(len(ids) * SPLIT_RATIOS[1]))
    for j, ctx_id in enumerate(ids):
        split_of_ctx[ctx_id] = "test" if j < n_test else "val" if j < n_test + n_val else "train"

splits = {"train": [], "val": [], "test": []}
for q in qa_all:
    splits[split_of_ctx[q["context_id"]]].append(q)

for name, items in splits.items():
    saved = Path(f"data/adilet/splits/qa_{LANG}_{name}.json")
    if saved.exists():
        assert [q["id"] for q in items] == [q["id"] for q in json.loads(saved.read_text(encoding="utf-8"))], f"{name}: разбиение не совпадает с {saved}"
ft_report = Path(f"data/adilet/finetune_results_{LANG}.json")
if ft_report.exists():
    expected = json.loads(ft_report.read_text(encoding="utf-8"))["split_sizes"]
    assert {n: len(v) for n, v in splits.items()} == expected, f"Размеры частей не совпадают с {ft_report}: {expected}"
print("Разбиение совпадает с finetune_e5.ipynb:", {n: len(v) for n, v in splits.items()})

if SMOKE_TEST:
    splits = {"train": splits["train"][:300], "val": splits["val"][:100], "test": splits["test"][:100]}

In [ ]:
chunks, chunk_ids, chunk_ctx = build_fixed_chunks(sub_df, CHUNK_SIZE, CHUNK_OVERLAP)
chunk_pos = {cid: i for i, cid in enumerate(chunk_ids)}
matcher = GoldChunkMatcher(chunks, chunk_ctx, min_coverage=MIN_COVERAGE, tie_margin=TIE_MARGIN)
for items in splits.values():
    for q in items:
        q["gold_idx"], q["gold_type"] = matcher.find(q)
gold = {name: [[chunk_ids[i] for i in q["gold_idx"]] for q in items] for name, items in splits.items()}

print(f"Чанков в корпусе: {len(chunks)}")
pd.DataFrame({name: Counter(q["gold_type"] for q in items) for name, items in splits.items()}).fillna(0).astype(int)

## 2. Первый этап: кандидаты гибрида

Для каждого вопроса — топ-50 Hybrid (RRF dense + BM25), ровно как в `eval_reranker.py`. Из них: на train — негативы
для обучения, на val/test — кандидаты для оценки. Строка «эталон в топ-N» — потолок, выше которого реранкер с пулом N
подняться не может.

In [ ]:
MAX_POOL = max(POOL_SIZES)
FIRST_POOL = max(MAX_POOL * 4, 20)   # пулы dense и BM25 перед слиянием — как в eval_reranker.py

t0 = time.time()
bm25 = BM25Retriever()
bm25.add_documents(chunks, chunk_ids)
embedder = EmbeddingModel(FIRST_STAGE_MODEL, device=DEVICE)
chunk_emb = get_or_compute_embeddings(
    embedder, chunks, CACHE_DIR,
    cache_key=f"{model_cache_name(FIRST_STAGE_MODEL)}_{LANG}_fixed_{CHUNK_SIZE}_{CHUNK_OVERLAP}",
)
index = VectorRetriever(embedding_dim=chunk_emb.shape[1])
index.add_documents(chunk_emb, chunks, chunk_ids)

candidates = {}
for name, items in splits.items():
    queries = [q["query"] for q in items]
    _, idx = index.search(embedder.encode_queries(queries), k=FIRST_POOL)
    dense = [[chunk_ids[i] for i in row if i >= 0] for row in idx]
    sparse = bm25.batch_search(queries, k=FIRST_POOL)   # BM25 — самая медленная часть (~3 с на 100 вопросов)
    candidates[name] = rrf_fuse(dense, sparse, HYBRID["dense_weight"], HYBRID["rrf_k"], MAX_POOL)
    print(f"{name}: {len(items)} вопросов, {time.time() - t0:.0f} с")
cand_idx = {name: [[chunk_pos[c] for c in row] for row in rows] for name, rows in candidates.items()}

del embedder, index, chunk_emb, bm25
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

hybrid_metrics = {name: ir_metrics(gold[name], [c[:EVAL_K] for c in candidates[name]]) for name in ["val", "test"]}
pool_recall = {
    name: {f"эталон в топ-{n}": np.mean([bool(set(g) & set(c[:n])) for g, c in zip(gold[name], candidates[name])]) for n in [10, 20, 30, 50]}
    for name in splits
}
print("Hybrid, test MRR@10:", round(hybrid_metrics["test"]["MRR@10"], 4), "(в reranker_eval_rus.json: 0.6501)")
pd.DataFrame(pool_recall).T.round(3)

## 3. Обучающие пары и hard negatives

Для каждого train-вопроса (кроме типа `context`, где эталон неточный):
- **позитивы** — все эталонные чанки (при почти равном покрытии их несколько, как в оценке), метка 1;
- **негативы** — `N_NEG` случайных чанков из топ-`NEG_DEPTH` гибрида, метка 0. Отбрасываются:
  эталоны и их дубликаты по тексту; чанки, содержащие большую долю слов ответа (вероятные ложные негативы).
  Соседние чанки того же фрагмента **допускаются** (в отличие от e5) с более строгим порогом: в работе реранкер
  должен поставить эталон выше соседей, и это самые трудные кандидаты;
- если подходящих меньше `N_NEG` — добор случайными чанками того же закона из других фрагментов.

`pos_weight` в функции потерь = негативов / позитивов, чтобы классы весили одинаково.

> Тонкость: e5 дообучалась на этих же train-вопросах, поэтому на train эталон в выдаче гибрида стоит выше,
> чем на test. Это не мешает: негативы всё равно — самые похожие на вопрос чужие чанки.

In [ ]:
rng_neg = np.random.default_rng(SEED)
chunk_idx_by_law = defaultdict(list)
for i, ctx_id in enumerate(chunk_ctx):
    chunk_idx_by_law[ctx_law[ctx_id]].append(i)

pairs = {"query": [], "passage": [], "label": []}
neg_stats = Counter()
for q, cands in zip(splits["train"], cand_idx["train"]):
    if q["gold_type"] == "context":
        neg_stats["вопросов пропущено (context)"] += 1
        continue
    gold_set = set(q["gold_idx"])
    gold_texts = {chunks[i] for i in gold_set}
    answer = matcher.stems(q.get("full_response", ""))

    pool = []
    for i in cands[:NEG_DEPTH]:
        if i in gold_set or chunks[i] in gold_texts:
            continue
        same_ctx = chunk_ctx[i] == q["context_id"]
        cov = len(answer & matcher.chunk_stems[i]) / len(answer) if answer else 0.0
        if cov >= (NEG_MAX_COV_SAME if same_ctx else NEG_MAX_COV_OTHER):
            neg_stats[f"отброшено как вероятный ложный негатив ({'тот же' if same_ctx else 'другой'} фрагмент)"] += 1
            continue
        pool.append(i)

    negs = [int(i) for i in rng_neg.choice(pool, size=min(N_NEG, len(pool)), replace=False)] if pool else []
    if len(negs) < N_NEG:
        neg_stats["вопросов с добором случайными"] += 1
        other = [i for i in chunk_idx_by_law[q["source_title"]] if chunk_ctx[i] != q["context_id"] and i not in negs]
        negs += [int(i) for i in rng_neg.choice(other, size=N_NEG - len(negs), replace=False)]
    neg_stats["негативов из того же фрагмента"] += sum(chunk_ctx[i] == q["context_id"] for i in negs)

    for text in dict.fromkeys(chunks[i] for i in q["gold_idx"]):
        pairs["query"].append(q["query"]); pairs["passage"].append(text); pairs["label"].append(1.0)
    for i in negs:
        pairs["query"].append(q["query"]); pairs["passage"].append(chunks[i]); pairs["label"].append(0.0)

n_pos = int(sum(pairs["label"]))
n_neg = len(pairs["label"]) - n_pos
POS_WEIGHT = n_neg / n_pos
print(f"Пар: {len(pairs['label'])} (позитивов {n_pos}, негативов {n_neg}), pos_weight = {POS_WEIGHT:.2f}")
for k, v in neg_stats.items():
    print(f"  {k}: {v}")
ex = splits["train"][0]
print("\nВопрос:", ex["query"])
print("Эталон:", chunks[ex["gold_idx"][0]][:200], "...")

## 4. Оценка: функция и baseline без дообучения

`rerank_eval` повторяет `eval_reranker.py`: реранкер один раз оценивает топ-50 гибрида, затем для каждого пула
N ∈ {10, 20, 30, 50} переставляются топ-N, а хвост идёт следом. Пул выбирается на **val** (наименьший в пределах 0,002
от лучшего MRR@10), test считается для итогового сравнения. Модель загружается через `src/reranker.py` — так же,
как в `ask.py` и веб-демо.

Baseline должен совпасть с `data/adilet/reranker_eval_{lang}.json` (с точностью до шума fp16).

In [ ]:
def free_gpu():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def rerank_eval(model_name_or_path):
    """Метрики реранкера по пулам на val/test и переранжированная выдача test при выбранном пуле."""
    reranker = CrossEncoderReranker(str(model_name_or_path), device=DEVICE, max_length=MAX_LEN)
    scores, ms = {}, {}
    for name in ["val", "test"]:
        t0 = time.time()
        scores[name] = [
            dict(reranker.rerank(q["query"], [(d, chunks[chunk_pos[d]]) for d in c]))
            for q, c in zip(splits[name], candidates[name])
        ]
        ms[name] = (time.time() - t0) / len(splits[name]) * 1000
    del reranker
    free_gpu()

    by_pool = {
        pool: {name: ir_metrics(gold[name], [p[:EVAL_K] for p in apply_rerank(candidates[name], scores[name], pool)])
               for name in ["val", "test"]}
        for pool in POOL_SIZES
    }
    best_pool = choose_pool({pool: by_pool[pool]["val"]["MRR@10"] for pool in POOL_SIZES})
    summary = {
        "best_pool": best_pool,
        "ms_per_query_at_max_pool": round(ms["test"], 1),
        "by_pool": {str(pool): m for pool, m in by_pool.items()},
    }
    return summary, apply_rerank(candidates["test"], scores["test"], best_pool)


def pool_table(summary):
    return pd.DataFrame({f"пул {p}": m["test"] for p, m in summary["by_pool"].items()}).T.round(4)


zero_shot, zero_shot_test_pred = rerank_eval(BASE_RERANKER)
print(f"{BASE_RERANKER} без дообучения: выбранный пул {zero_shot['best_pool']}, "
      f"{zero_shot['ms_per_query_at_max_pool']} мс на вопрос (пул {MAX_POOL})")
prev = Path(f"data/adilet/reranker_eval_{LANG}.json")
if prev.exists() and not SMOKE_TEST:
    old = json.loads(prev.read_text(encoding="utf-8"))["rerankers"].get(BASE_RERANKER)
    if old:
        p = str(old["best_pool"])
        print(f"Сверка с {prev}: test MRR@10 при пуле {p}: было {old['by_pool'][p]['test']['MRR@10']:.4f}, "
              f"сейчас {zero_shot['by_pool'][p]['test']['MRR@10']:.4f}")
pool_table(zero_shot)

## 5. Дообучение

- `CrossEncoderTrainer` + `BinaryCrossEntropyLoss(pos_weight)` — стандартный рецепт обучения реранкеров
  на hard negatives в `sentence-transformers`.
- Во время обучения `CrossEncoderRerankingEvaluator` считает MRR@10 на val, переставляя топ-20 гибрида
  (вопросы без эталона в топ-20 он пропускает, поэтому его число выше итогового — оно нужно только для выбора точки).
  Оценка `EVALS_PER_EPOCH` раз за эпоху, в конце загружается лучшая точка.
- Промежуточные чекпоинты (с состоянием оптимизатора, для bge ~7 ГБ) и модели запусков пишутся на локальный диск
  Colab; на Drive в конце копируется только лучший по val запуск, плюс журналы всех запусков.
- В пределах сессии цикл возобновляемый: если `CHECKPOINT_DIR/seed_N/model` уже есть (например, ячейку прервали
  на следующем seed), запуск не обучается заново, а только оценивается. После обрыва сессии обучение начинается заново.

In [ ]:
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

val_evaluator = CrossEncoderRerankingEvaluator(
    samples=[
        {"query": q["query"], "positive": [chunks[i] for i in q["gold_idx"]], "documents": [chunks[i] for i in c[:EVAL_POOL]]}
        for q, c in zip(splits["val"], cand_idx["val"])
    ],
    at_k=10,
    always_rerank_positives=False,   # только кандидаты гибрида, как в реальной работе
    name="val",
    batch_size=64,
)

steps_per_epoch = math.ceil(len(pairs["label"]) / (P["batch_size"] * P["grad_accum"]))
eval_steps = max(1, steps_per_epoch // EVALS_PER_EPOCH)
print(f"Шагов на эпоху: {steps_per_epoch}, оценка каждые {eval_steps} шагов")

runs, test_preds = {}, {}
for train_seed in TRAIN_SEEDS:
    print(f"\n===== seed {train_seed} =====")
    run_dir = CHECKPOINT_DIR / f"seed_{train_seed}"
    model_dir = run_dir / "model"

    if (model_dir / "config.json").exists():
        print(f"Модель уже обучена: {model_dir} — только оценка")
    else:
        tmp_dir = TRAINER_TMP / f"seed_{train_seed}"
        shutil.rmtree(tmp_dir, ignore_errors=True)
        model = CrossEncoder(BASE_RERANKER, device=DEVICE, max_length=MAX_LEN)
        if P["grad_ckpt"]:
            # Включаем у модели напрямую: Trainer из Transformers v5 передаёт every_n_layers / offload,
            # которых нет в CrossEncoder.gradient_checkpointing_enable (sentence-transformers 6.1) → TypeError
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        train_dataset = Dataset.from_dict(pairs).shuffle(seed=train_seed)

        args = CrossEncoderTrainingArguments(
            output_dir=str(tmp_dir),
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=P["batch_size"],
            gradient_accumulation_steps=P["grad_accum"],
            learning_rate=P["lr"],
            warmup_steps=WARMUP_RATIO,          # в Transformers v5 дробное значение означает долю шагов
            fp16=DEVICE == "cuda" and not USE_BF16,
            bf16=USE_BF16,
            eval_strategy="steps",
            eval_steps=eval_steps,
            save_strategy="steps",
            save_steps=eval_steps,
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="eval_val_mrr@10",
            greater_is_better=True,
            logging_steps=50,
            seed=train_seed,
            dataloader_num_workers=2 if IN_COLAB else 0,
            report_to="none",
        )
        trainer = CrossEncoderTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            loss=BinaryCrossEntropyLoss(model, pos_weight=torch.tensor(POS_WEIGHT, device=model.device)),
            evaluator=val_evaluator,
        )
        print("До обучения:", {k: round(v, 4) for k, v in val_evaluator(model).items() if "mrr" in k})
        trainer.train()

        run_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(model_dir))
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "history.csv", index=False)
        del trainer, model
        free_gpu()
        shutil.rmtree(tmp_dir, ignore_errors=True)

    runs[train_seed], test_preds[train_seed] = rerank_eval(model_dir)
    (run_dir / "metrics.json").write_text(json.dumps(runs[train_seed], ensure_ascii=False, indent=2), encoding="utf-8")
    # Журнал и метрики запуска — на Drive (модель остаётся на локальном диске)
    log_dir = LOG_DIR / f"seed_{train_seed}"
    log_dir.mkdir(parents=True, exist_ok=True)
    for fname in ["metrics.json", "history.csv"]:
        if (run_dir / fname).exists() and (run_dir / fname).resolve() != (log_dir / fname).resolve():
            shutil.copy2(run_dir / fname, log_dir / fname)
    best = runs[train_seed]["by_pool"][str(runs[train_seed]["best_pool"])]
    print(f"seed {train_seed}: пул {runs[train_seed]['best_pool']}, "
          f"val MRR@10 {best['val']['MRR@10']:.4f}, test MRR@10 {best['test']['MRR@10']:.4f}")

In [ ]:
# Динамика обучения: MRR@10 на val (оценщик, топ-20) и средний loss между оценками
for train_seed in runs:
    hist_path = CHECKPOINT_DIR / f"seed_{train_seed}" / "history.csv"
    if not hist_path.exists():
        continue
    h = pd.read_csv(hist_path)
    val_log = h.dropna(subset=["eval_val_mrr@10"])[["step", "epoch", "eval_val_mrr@10"]].set_index("step")
    loss = h.dropna(subset=["loss"])
    val_log["loss"] = [loss[loss["step"] <= s]["loss"].tail(max(1, eval_steps // 50)).mean() for s in val_log.index]
    print(f"seed {train_seed}")
    display(val_log.round(4))

## 6. Сравнение «до / после» и сохранение

In [ ]:
def at_best_pool(summary, split):
    return summary["by_pool"][str(summary["best_pool"])][split]


best_seed = max(runs, key=lambda s: at_best_pool(runs[s], "val")["MRR@10"])
METRICS = list(hybrid_metrics["test"])
ft_values = pd.DataFrame([at_best_pool(runs[s], "test") for s in runs])
ft_mean, ft_std = ft_values.mean(), ft_values.std(ddof=0)

rows = {
    "Hybrid (первый этап)": {m: f"{hybrid_metrics['test'][m]:.4f}" for m in METRICS},
    f"+ {BASE_RERANKER} без дообучения (пул {zero_shot['best_pool']})": {m: f"{at_best_pool(zero_shot, 'test')[m]:.4f}" for m in METRICS},
    f"+ дообученный ({len(runs)} seed)": {
        m: f"{ft_mean[m]:.4f}" + (f" ± {ft_std[m]:.4f}" if len(runs) > 1 else "") for m in METRICS
    },
}
print(f"Test, {len(splits['test'])} вопросов. Лучший по val запуск: seed {best_seed}, пул {runs[best_seed]['best_pool']}")
display(pd.DataFrame(rows).T)
print("Дообученный реранкер (лучший seed), test по пулам:")
pool_table(runs[best_seed])

In [ ]:
# Лучший по val запуск — в OUTPUT_DIR; отчёт дополняет файл (ключ — исходная модель), чтобы хранить и bge, и mMiniLM
if not SMOKE_TEST:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    shutil.copytree(CHECKPOINT_DIR / f"seed_{best_seed}" / "model", OUTPUT_DIR)

    report = json.loads(RESULTS_PATH.read_text(encoding="utf-8")) if RESULTS_PATH.exists() else {}
    report.update({
        "language": LANG,
        "first_stage": {"method": "hybrid", "model_name": Path(FIRST_STAGE_MODEL).as_posix(), **HYBRID},
        "selection_metric": "val MRR@10",
        "selection_rule": f"пул — наименьший в пределах {POOL_TOLERANCE} от лучшего; запуск — лучший по val",
        "pool_sizes": POOL_SIZES,
        "split_sizes": {name: len(items) for name, items in splits.items()},
        "hybrid": hybrid_metrics,
    })
    report.setdefault("rerankers", {})[BASE_RERANKER] = {
        "finetuned_model": f"models/{RUN_NAME}",
        "best_seed_by_val": best_seed,
        "train": {"questions": len(splits["train"]) - neg_stats["вопросов пропущено (context)"],
                  "pairs": len(pairs["label"]), "positives": n_pos, "negatives": n_neg,
                  "pos_weight": POS_WEIGHT, "neg_stats": dict(neg_stats)},
        "hyperparams": {**P, "epochs": EPOCHS, "evals_per_epoch": EVALS_PER_EPOCH, "warmup_ratio": WARMUP_RATIO,
                        "max_len": MAX_LEN, "eval_pool": EVAL_POOL, "neg_depth": NEG_DEPTH, "n_neg": N_NEG,
                        "neg_max_cov_other": NEG_MAX_COV_OTHER, "neg_max_cov_same": NEG_MAX_COV_SAME,
                        "precision": "bf16" if USE_BF16 else "fp16" if DEVICE == "cuda" else "fp32",
                        "gpu": gpu, "train_seeds": TRAIN_SEEDS},
        "zero_shot": zero_shot,
        "finetuned_runs": {str(s): runs[s] for s in runs},
        "finetuned_mean_test": ft_mean.to_dict(),
        "finetuned_std_test": ft_std.to_dict(),
    }
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Модель: {OUTPUT_DIR}\nОтчёт:  {RESULTS_PATH}")

In [ ]:
# Примеры: без дообучения эталон не на 1-м месте, после дообучения — на 1-м
ft_pred = test_preds[best_seed]
improved = [
    (q, b, f) for q, b, f in zip(splits["test"], zero_shot_test_pred, ft_pred)
    if b and f and chunk_pos[b[0]] not in q["gold_idx"] and chunk_pos[f[0]] in q["gold_idx"]
]
worse = [
    (q, b, f) for q, b, f in zip(splits["test"], zero_shot_test_pred, ft_pred)
    if b and f and chunk_pos[b[0]] in q["gold_idx"] and chunk_pos[f[0]] not in q["gold_idx"]
]
print(f"Эталон стал 1-м: {len(improved)}, перестал быть 1-м: {len(worse)} (из {len(splits['test'])})\n")
for q, b, f in improved[:5]:
    print(f"[{q['source_title']}] {q['query']}")
    print(f"  без дообучения топ-1: {chunks[chunk_pos[b[0]]][:150]}...")
    print(f"  после дообучения:     {chunks[chunk_pos[f[0]]][:150]}...\n")

## Дальнейшие шаги

1. Скачать с Drive в проект (пути те же): `models/reranker-…-adilet-rus/` и `data/adilet/reranker_finetune_rus.json`.
2. Проверить локально в той же процедуре, что и остальные реранкеры:
   `python eval_reranker.py --lang rus --rerankers models/reranker-bge-m3-adilet-rus BAAI/bge-reranker-v2-m3`
3. Поиск с дообученным реранкером: `python ask.py --rerank --reranker models/reranker-bge-m3-adilet-rus`.
   Веб-демо пока берёт `DEFAULT_RERANKER` из `src/reranker.py` — его нужно переключить на дообученную модель.
4. Запустить ещё раз с `BASE_RERANKER = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"`: если после дообучения
   он близок к bge, это лучший вариант для демо (в ~7 раз быстрее).
5. Для отчёта: `TRAIN_SEEDS = [42, 43, 44]` — разброс по seed, как у e5.